# Stage 2 end to end (README §4.4, Phase 11): request -> [contract] -> code -> check -> repair (GPU)

Runs `eval/runner_stage2.py` for each Stage 2 system over the tasks and judges every final output against the reference.
Scoring runs on the laptop: download `stage2_bundle_<stage>.zip` (link printed by the last cell), unzip it at the project
root, then `python eval/stage2_metrics.py results/stage2 <stage>`.

| Preset | Contract | Generator sees contract | Checker | Repair | Needs |
|---|---|---|---|---|---|
| `baseline1` | none | no | none | none | — |
| `baseline2_repair` | none | no | conventional | agent | — |
| `baseline3_repair` | prompted 7B (p1-v3) | no | intent | agent | `kaggle_outputs/parsing/7b_p1-v3/parses.jsonl` (in the repo) |
| `gold_contract` | gold | yes (b2-v1) | intent | agent | — |
| `proposed` | fine-tuned parser | yes (b2-v1) | intent | agent | `FT_PARSES` |
| `no_contract_to_generator` | fine-tuned parser | no | intent | agent | `FT_PARSES` |

**Step 1 (`STAGE = "dev"`, 20 dev tasks):** `baseline1` and `gold_contract`. This checks the new contract-aware generator
prompt b2-v1 with the real model: compare `initial_code_accuracy` of the two (same generator, with and without the contract),
read a few `generator_reply` entries in `traces.jsonl`, and check for empty or non-code replies. Change b2-v1 only on dev
evidence, and give a changed prompt a new version.
**Step 2 (`STAGE = "test"`, 100 tasks):** the systems in `SYSTEMS`. `proposed` and `no_contract_to_generator` are skipped
until `FT_PARSES` points to the fine-tuned parser's `parses.jsonl`.

Models: code generator `GENERATOR_MODEL` (the pool's Qwen2.5-3B, fp16) and repair agent `AGENT_MODEL` (Qwen2.5-7B, 4-bit),
the same for every system. Both stay on the GPU (about 6 + 5.5 GB on a T4). Every run is resumable and cached; a
results folder refuses runs with a different configuration.

Settings: GPU T4 x1, Internet on, secret `GITHUB_TOKEN` for a private repo. *Save Version → Save & Run All*.

**GPU time (rough, not measured yet):** one generator call per task plus up to 3 repair attempts (several agent steps
each, ~15 s per 7B step) when the first code fails, so about 1–3 h per system on test; `baseline1` is fast (no agent).
Run fewer systems per session if needed: it resumes.

In [ ]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/<user>/<repo>.git"   # <- your repo
BRANCH = "main"
STAGE = "dev"                  # "dev" first (b2-v1 prompt check), then "test"
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"; GENERATOR_4BIT = False    # the pool's generator; same for every system
AGENT_MODEL = "Qwen/Qwen2.5-7B-Instruct"; AGENT_4BIT = True             # the frozen repair agent (a1-v4)
B3_PARSES = "kaggle_outputs/parsing/7b_p1-v3/parses.jsonl"             # Baseline 3 contracts (prompted 7B, p1-v3)
FT_PARSES = None               # e.g. "kaggle_outputs/parsing/ft_1_5b/parses.jsonl" once the fine-tuned parser has run
FT_TAG = "ft_1_5b"             # names the results folders of the fine-tuned systems (one folder per parser)
SEEDS = [0]                    # temperature 0 (greedy): extra seeds give identical runs; use >1 only with TEMPERATURE > 0
TEMPERATURE = 0.0
BACKEND = "hf"                 # "pipeline_check" = scripted models, for testing the notebook only (NOT results)
SYSTEMS = ["baseline1", "gold_contract"] if STAGE == "dev" else \
          ["baseline1", "baseline2_repair", "baseline3_repair", "gold_contract", "proposed", "no_contract_to_generator"]
WORK = "/kaggle/working"
OUT = f"{WORK}/results/stage2"
CACHE = f"{WORK}/llm_cache"
GEN_TAG = GENERATOR_MODEL.split("/")[-1].replace("Qwen2.5-", "").replace("-Instruct", "").lower().replace(".", "_")
print(STAGE, SYSTEMS, "| generator", GENERATOR_MODEL, "| agent", AGENT_MODEL, "| FT parses", FT_PARSES)

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
NEEDED = ["eval/runner_stage2.py", "eval/stage2_metrics.py", "src/igt/agents/code_generator.py",
          "src/igt/agents/repair_agent.py", "results/stage1/baseline2/thresholds.json", B3_PARSES]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox + bitsandbytes ----------------
# The sandbox and verifier need pandas 2.3.3 / numpy 2.1.3. They go into a separate folder used only by this repo's
# processes, so Kaggle's torch stack stays untouched (same setup as the agent notebook).
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers, bitsandbytes; "
         "print('pandas', pandas.__version__, '| numpy', numpy.__version__, '| torch', torch.__version__, "
         "'| transformers', transformers.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "numpy 2.1.3" in p.stdout and "GPU True" in p.stdout, \
    "environment check failed: see the output above"

In [ ]:
# ---------------- Run ----------------
LOGS = f"{OUT}/logs"
os.makedirs(LOGS, exist_ok=True)
NEEDS_FT = {"proposed", "no_contract_to_generator"}

def system_name(preset):
    name = f"{preset}__gen_{GEN_TAG}"                 # one folder per (system, generator)
    return f"{name}__{FT_TAG}" if preset in NEEDS_FT else name

def run(preset):
    if preset in NEEDS_FT and not FT_PARSES:
        print(f"== skip {preset}: set FT_PARSES to the fine-tuned parser's parses.jsonl"); return
    system = system_name(preset)
    cmd = ["python", "eval/runner_stage2.py", "--preset", preset, "--system", system, "--backend", BACKEND,
           "--generator-model", GENERATOR_MODEL, "--agent-model", AGENT_MODEL, "--splits", STAGE,
           "--seeds", *map(str, SEEDS), "--temperature", str(TEMPERATURE), "--cache-dir", CACHE, "--out", OUT]
    cmd += ["--generator-4bit"] if GENERATOR_4BIT else []
    cmd += ["--agent-4bit"] if AGENT_4BIT else []
    if preset == "baseline3_repair":
        cmd += ["--parses", B3_PARSES]
    if preset in NEEDS_FT:
        cmd += ["--parses", FT_PARSES]
    t0 = time.time()
    p = subprocess.run(cmd, env=ENV, capture_output=True, text=True)
    with open(f"{LOGS}/{system}_{STAGE}.log", "a") as fh:          # full stdout/stderr, kept in the bundle
        fh.write(f"$ {' '.join(cmd)}\n--- stdout ---\n{p.stdout}\n--- stderr ---\n{p.stderr}\n"
                 f"--- exit {p.returncode}, {(time.time() - t0) / 60:.1f} min ---\n")
    print(p.stdout[-2500:]); print(p.stderr[-2500:])
    assert p.returncode == 0, f"{system} failed: see {LOGS}/{system}_{STAGE}.log"
    print(f"== {system} done in {(time.time() - t0) / 60:.1f} min")

for preset in SYSTEMS:
    run(preset)

In [ ]:
# ---------------- Summary + ONE zip with everything needed on the laptop ----------------
# Unzip at the project root: results land in results/stage2/<system>/ (runs.jsonl, traces.jsonl, run_info.json,
# git_commit.txt), plus results/stage2/stage2_metrics_<stage>.json, metrics_<stage>.txt, logs/ and bundle_info.json.
import zipfile, datetime
CLEANUP_REPO = False      # True for "Save Version" runs; keep False in an interactive session so cells can be rerun

p = subprocess.run(["python", "eval/stage2_metrics.py", OUT, STAGE], env=ENV, capture_output=True, text=True)
print(p.stdout[-4000:], p.stderr[-1500:])
open(f"{OUT}/metrics_{STAGE}.txt", "w").write(p.stdout + "\n" + p.stderr)
if STAGE == "dev":        # b2-v1 check: same generator, with vs without the contract (first code, before any repair)
    m = json.load(open(f"{OUT}/stage2_metrics_{STAGE}.json"))["systems"]
    for k, v in m.items():
        print(f"{k:<45} initial code accuracy {v['initial_code_accuracy']}  final {v['final_accuracy']}")

systems = sorted(d for d in os.listdir(OUT) if os.path.isdir(f"{OUT}/{d}") and d != "logs")
for d in systems:
    open(f"{OUT}/{d}/git_commit.txt", "w").write(COMMIT)
info = dict(stage=STAGE, commit=COMMIT, generator_model=GENERATOR_MODEL, generator_4bit=GENERATOR_4BIT,
            agent_model=AGENT_MODEL, agent_4bit=AGENT_4BIT, ft_parses=FT_PARSES, seeds=SEEDS, backend=BACKEND,
            systems=systems, created=datetime.datetime.now().isoformat(timespec="seconds"),
            runs={d: sum(1 for _ in open(f"{OUT}/{d}/runs.jsonl")) for d in systems if os.path.exists(f"{OUT}/{d}/runs.jsonl")})
json.dump(info, open(f"{OUT}/bundle_info.json", "w"), indent=2)

BUNDLE = f"{WORK}/stage2_bundle_{STAGE}.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.join("results/stage2", os.path.relpath(full, OUT)))
    names = z.namelist()
print(f"saved {BUNDLE}: {len(names)} files, {os.path.getsize(BUNDLE) / 1e6:.1f} MB")
print("\n".join("  " + n for n in names))
print(json.dumps(info, indent=2))
try:
    from IPython.display import FileLink, display
    os.chdir(WORK)
    display(FileLink(os.path.basename(BUNDLE)))
except Exception:
    pass
if CLEANUP_REPO:
    os.chdir(WORK)
    shutil.rmtree(f"{WORK}/repo", ignore_errors=True)